In [ ]:
import numpy as np
import pandas as pd
#import scipy.stats as sps
import seaborn as sns
from scipy import stats
from statsmodels.stats.power import tt_ind_solve_power

# Кейс
- Вы работаете аналитиком команды монетизации. Тестируете новую систему цен на очень небольшую аудиторию - Профессионалы в небольшом городе.
- Аудитория достаточно пассивная, поэтому в ней редко "прокрашиваются" фичи. Каждая потенциальная фича на вес золота.
- Менеджеры имеют большие надежды на эту фичу.
- Основная метрика ARPPU. Даже такое небольшое повышение позволит начать зарабатыват больше.

**Вам нужно**
- провести первичный анализ и найти наблюдаемый эффект между группами
- проанализировать базовый АБ-тест и понять есть ли стат значимое изменение в post данных после внедрения новых цен
- провести CUPED t-test
- рассчитать размер выборки для CUPED t-test

## Возмем нужные функции из кода лекции

In [ ]:
def generate_corr_data(N, pre_mean, post_mean, corr = 0.8):
    """Генерируем датафрейм с нужной корреляцией между 2 ковариатами.

    N - размер data frame
    pre_mean - средняя до старта теста
    post_mean - средняя после старта теста
    corr - корреляция метрики во время эксперимента с метрикой до эксперимента.
    Возвращает датафрейм с коррелированной метрикой до эксперимента и после эксперимента
    """
    cov = [[1, corr], [corr, 1]]
    correlated_data = np.random.multivariate_normal([pre_mean, post_mean], cov, N)
    correlated_df = pd.DataFrame({'post_value': correlated_data[:, 1], 'pre_value': correlated_data[:, 0]})
    return (correlated_df)

def calculate_theta_basic(pre, post):
    theta = (np.cov(post, pre)[0, 1]) /\
            (np.var(pre))
    return(theta)

def calculate_theta_for_test(control_pre, control_post, test_pre, test_post):
    theta = (np.cov(control_post, control_pre)[0, 1] + np.cov(test_post, test_pre)[0, 1]) /\
            (np.var(control_pre) + np.var(test_pre))
    return(theta)

In [ ]:
def get_basic_ttest(group_A, group_B):
    '''Проверяет гипотезу о равенстве средних для обычного среднего.
    return - t_stat, p_value'''

    t_stat, p_value = stats.ttest_ind(group_A, group_B)
    inference = {'t_stat': t_stat, 'p_value':p_value}
    return(inference)

def get_cuped_ttest(control_pre, control_post, test_pre, test_post):
    '''Проверяет гипотезу о равенстве средних CUPED вариант
    return - t_stat, p_value'''

    theta = calculate_theta_for_test(control_pre, control_post, test_pre, test_post)

    control_cuped = control_post - theta * control_pre
    test_cuped = test_post - theta * test_pre

    inference = get_basic_ttest(control_cuped, test_cuped)

    return(inference)

# 1. Загружаем данные

In [ ]:
# загружаем данные по тесту
data = pd.read_csv('cuped_homework.csv')
data

,post_ARPPU,pre_ARPPU,group
0,660.0,595.0,A
1,540.0,621.0,A
2,863.0,782.0,A
3,431.0,567.0,A
4,434.0,473.0,A
...,...,...,...
4455,628.0,679.0,B
4456,758.0,634.0,B
4457,586.0,612.0,B
4458,698.0,791.0,B


## 1. Первичный анализ
- Задание 1.1. Укажите наблюдаемую абсольтную разницу между средними у Б и А групп.
- Задание 1.2. Укажите наблюдаемый effect size Cohen D. Для рассчетов используйте асолютную разницу и стандартное отклонение в pre данных в группе А

In [ ]:
# посмотрим на средние по группам
agg_df = data.groupby('group').mean()
agg_df

,post_ARPPU,pre_ARPPU
group,,
A,598.689905,596.972149
B,604.045757,597.265784


In [ ]:
abs_observed_lift = agg_df['post_ARPPU']['B'] - agg_df['post_ARPPU']['A']
print('Задание 1.1. Абсолютный наблюдаемый эффект =', abs_observed_lift)

cohen_d = abs_observed_lift / data['pre_ARPPU'][data['group']=='A'].std()
print('Задание 1.2. Наблюдаемый Cohen D размер эффекта =', cohen_d)

Задание 1.1. Абсолютный наблюдаемый эффект = 5.355852506777865
Задание 1.2. Наблюдаемый Cohen D размер эффекта = 0.054157164525994464


## 2. Проведите простой t-test для POST данных
- Задание 2.1 Укажите p-value полученное при проведении простого t-test на post данных

In [ ]:
# Задание 2.1 Укажите p-value полученное при проведении простого t-test на post данных
# проведем простой t-test. Найдем p_value
inference =  get_basic_ttest(data['post_ARPPU'][data['group'] == 'A'], data['post_ARPPU'][data['group'] == 'B'])
print('Задание 2.1. p-value простого t-test =', round(inference['p_value'], 3))

Задание 2.1. p-value простого t-test = 0.074


- Видим, что t-test не показал стат. значимое различие. Можно расстраивать  менеджера?
- Давайте поробуем использовать pre данные. Вдруг они помогут повысить чувствительность теста.

## 3. Проведите Cuped t-test
- Задание 3.1. Укажите долю пустых значений в pre данных
- Задание 3.2. Укажите p-value в пооведенном cuped t-test

In [ ]:
# Задание 3.1. Укажите долю пустых значений в pre данных
print('Задание 3.1. Доля пустых значений в pre данных равна =', round(data['pre_ARPPU'].isna().sum()/ len(data['pre_ARPPU']), 3))

Задание 3.1. Доля пустых значений в pre данных равна = 0.133


In [ ]:
# заполним пустые значения средней на pre периоде
pre_mean = data['pre_ARPPU'].mean()
data['pre_ARPPU'].fillna(pre_mean, inplace = True)

In [ ]:
# разобьем на соответствующие группы для того чтобы провести cuped t-test
pre_control = data['pre_ARPPU'][data['group'] == 'A']
post_control = data['post_ARPPU'][data['group'] == 'A']
pre_test = data['pre_ARPPU'][data['group'] == 'B']
post_test = data['post_ARPPU'][data['group'] == 'B']

In [ ]:
# Проведем CUPED t-test
basic_ttest_inference = get_basic_ttest(data['post_ARPPU'][data['group'] == 'A'], data['post_ARPPU'][data['group'] == 'B'])
cuped_ttest_inference = get_cuped_ttest(pre_control, post_control, pre_test, post_test)

In [ ]:
# Задание 3.2. Укажите p-value в пооведенном cuped t-test
print('Задание 3.2. p-value в cuped t-test', round(cuped_ttest_inference['p_value'], 3))

Задание 3.2. p-value в cuped t-test 0.032


# 4. Рассчитать размер обычной и CUPED выборки
- альфа = 0.05 и мощности 0.8
- двусторонняя проверка гипотезы
- размер выборок 50/50
- статистический критерий t-test
- относительный MDE = 0.01
- за основу будем брать другой датасет на исторических данных cuped_homework_pre_pre_data.csv. он имеет похожу схему. только отсутствует деление на группы
- не забываем заполнить пустые занчения средней прежде чем рассчитывать размер группы для CUPED/ Для рассчета обычной выборки пожно использовать post данные на истрическом периоде

**Задания:**
- Задание 4.1. Укажите чему равна theta по важшим рассчетам
- Задание 4.2. Укажите на сколько процентов стандартное отклонение Y_cuped меньше стандартного отклоения post_ARPPU
- Задание 4.3. Укажите размер выборки нужный для детекции такого MDE на post данных простым t-test
- Задание 4.4. Укажите размер выборки нужный для детекции такого MDE CUPED t-test

In [ ]:
# загружаем данные
pre_pre_data = pd.read_csv('cuped_homework_pre_pre_data.csv')
pre_pre_data

,post_ARPPU,pre_ARPPU
0,629.0,620.0
1,536.0,619.0
2,503.0,596.0
3,649.0,578.0
4,667.0,752.0
...,...,...
6505,560.0,NaN
6506,738.0,655.0
6507,664.0,728.0
6508,641.0,634.0


In [ ]:
# рассчитаем размер выборки нужный для базового теста

# заполним пустые значения средней на pre периоде
pre_mean = pre_pre_data['pre_ARPPU'].mean()
pre_pre_data['pre_ARPPU'].fillna(pre_mean, inplace = True)

# рассчитаем контрольные средние и std. Берем post период для обычной выборки
control_std = pre_pre_data['post_ARPPU'].std() # в функции generate_corr_data генерируются нормальные величины с std = 1
control_mean = pre_pre_data['post_ARPPU'].mean()
rel_effect = 0.01
mean_diff = control_mean * rel_effect
alpha = 0.05
power = 0.8


# Расчет индекса Коэна
cohen_d_basic  = mean_diff / control_std
print(cohen_d_basic)
n = tt_ind_solve_power(effect_size = cohen_d_basic,
                       alpha = alpha,
                       power = power,
                       ratio = 1,
                       alternative = "two-sided")
basic_sample_size = round(n)
print('Basic sample size =', basic_sample_size)

theta = calculate_theta_basic(pre_pre_data['pre_ARPPU'], pre_pre_data['post_ARPPU'])
Y_cuped = pre_pre_data['post_ARPPU'] - theta * pre_pre_data['pre_ARPPU']
Y_cuped_std = Y_cuped.std()

cohen_d_cuped  = mean_diff / Y_cuped_std
print(cohen_d_cuped)
n_cuped = tt_ind_solve_power(effect_size = cohen_d_cuped,
                       alpha = alpha,
                       power = 0.8,
                       ratio = 1,
                       alternative = "two-sided")
cuped_sample_size = round(n_cuped)
print('CUPED sample size', cuped_sample_size)

0.05924643319868699
Basic sample size = 4473
0.07186462188408765
CUPED sample size 3040


In [ ]:
# Задание 4.1. Укажите чему равна theta по важшим рассчетам
print('Задание 4.1. Theta равна = ', theta)

Задание 4.1. Theta равна =  0.6096554199736017


In [ ]:
# Задание 4.2. Укажите на сколько процентов стандартное отклонение Y_cuped меньше стандартного отклоения post_ARPPU
rel_diff_in_std = abs(Y_cuped_std / control_std - 1)
print('Задание 4.2. Cтандартное отклонение Y_cuped меньше стандартного отклоения post_ARPPU на {} %'.format(round(rel_diff_in_std*100, 1)))

Задание 4.2. Cтандартное отклонение Y_cuped меньше стандартного отклоения post_ARPPU на 17.6 %


In [ ]:
# Задание 4.3. Укажите размер выборки нужный для детекции такого MDE на post данных простым t-test
print('Задание 4.3. Размер выборки для обычного t-test =', basic_sample_size)

Задание 4.3. Размер выборки для обычного t-test = 4473


In [ ]:
# Задание 4.4. Укажите размер выборки нужный для детекции такого MDE CUPED t-test
print('Задание 4.4. Размер выборки для CUPED t-test =', cuped_sample_size)

Задание 4.4. Размер выборки для CUPED t-test = 3040
